# 04 — Revised ML Eligibility + Background Candidate Generation

**FYP:** Developing a Semi-Dynamic Landslide Susceptibility Decision-Support Dashboard Using Machine Learning for Malaysia

This notebook **replaces the previous Stage 04 and Stage 04B notebooks for future runs**.

It does four things:

1. Preserves the full reviewed historical landslide inventory.
2. Creates a transparent **point-ML-eligible** subset based on positional accuracy.
3. Generates a larger, spatially distributed **background-candidate pool** rather than immediately declaring final pseudo-absences.
4. Applies only defensible candidate validity rules at this stage.

---

## Important modelling principle

Environmental predictors must **not** be used to deliberately choose only "safe-looking" negatives.

Therefore we do **not** require background candidates to be:
- on low slopes,
- far from roads,
- far from waterways,
- on low elevations,
- on supposedly stable geology.

Doing so would make classification artificially easy and bias model accuracy.

At this stage the valid exclusion rules are:

- outside Malaysia → exclude
- inside known-positive uncertainty/exclusion zone → exclude
- open water / invalid raster pixel → exclude later when a reliable mask is available
- missing required predictor coverage → exclude later during feature extraction
- too close to another background point → reject for spatial spread

Predictor values such as slope, elevation, geology, road distance and waterway distance are extracted **after candidate generation**.

---

## Temporal-feature principle

Factors are separated into:

### Static / quasi-static
- elevation
- slope
- aspect
- soil
- lithology
- distance to major fault

### Time-sensitive
- land cover
- roads
- rainfall

Historical values must be aligned with the event/reference date whenever possible.

For rainfall:
- full event date → event-specific antecedent rainfall may be calculated
- month+year only → monthly rainfall may be used as a *monthly contextual feature*, not labelled as 7/30-day antecedent rainfall
- year only → do not invent a month/day
- no reliable date → no event-specific rainfall; optionally use long-term climatological rainfall as a separate static climate variable

Background controls used in a rainfall-enhanced model must later be **date-matched to positive events**.


In [1]:
# OPTIONAL installation
# !pip install -q geopandas pandas numpy matplotlib shapely pyproj

from pathlib import Path
import re
import math
import warnings

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

from shapely.geometry import Point, box
from shapely.ops import unary_union

warnings.filterwarnings("ignore", category=UserWarning)

print("pandas:", pd.__version__)
print("geopandas:", gpd.__version__)


pandas: 3.0.2
geopandas: 1.1.4


## 1. Project paths

Change only `PROJECT_ROOT`.

This revised notebook uses the **Stage-3 final reviewed inventory** as its source, so old pseudo-absence outputs are not required.


In [2]:
PROJECT_ROOT = Path(r"C:\Landslide_FYP")

RAW_DIR = PROJECT_ROOT / "Landslide_FYP_Data"
DEV_DIR = PROJECT_ROOT / "Landslide_FYP_Development"

POSITIVE_PATH = (
    DEV_DIR / "02_cleaned_inventory"
    / "malaysia_positive_inventory_final.geojson"
)

BOUNDARY_PATH = (
    RAW_DIR / "02_Boundary"
    / "mys_admin0.geojson"
)

OUTPUT_DIR = (
    DEV_DIR / "01_preprocessing"
    / "stage04_revised_outputs"
)

CLEAN_DIR = DEV_DIR / "02_cleaned_inventory"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CLEAN_DIR.mkdir(parents=True, exist_ok=True)

for name, path in {
    "Stage-3 final positive inventory": POSITIVE_PATH,
    "Malaysia boundary": BOUNDARY_PATH,
}.items():
    print(f"{name}: {path}")
    if not path.exists():
        raise FileNotFoundError(path)


Stage-3 final positive inventory: C:\Landslide_FYP\Landslide_FYP_Development\02_cleaned_inventory\malaysia_positive_inventory_final.geojson
Malaysia boundary: C:\Landslide_FYP\Landslide_FYP_Data\02_Boundary\mys_admin0.geojson


## 2. Load the reviewed positive inventory

In [3]:
positive = gpd.read_file(POSITIVE_PATH).to_crs("EPSG:4326")
malaysia = gpd.read_file(BOUNDARY_PATH).to_crs("EPSG:4326")

print("Reviewed historical positives:", len(positive))
print()
display(
    positive["source_dataset"]
    .value_counts(dropna=False)
    .rename_axis("source_dataset")
    .reset_index(name="records")
)

if "final_positive_id" not in positive.columns:
    raise KeyError("final_positive_id missing from Stage-3 inventory.")

if positive["final_positive_id"].duplicated().any():
    raise ValueError("Duplicate final_positive_id values found.")


Reviewed historical positives: 287



,source_dataset,records
0,NASA_COOLR,235
1,JMG,52


## 3. Recalculate point-ML eligibility

The earlier ~200 count was a working result, not a permanent assumption.

Baseline rule:

- JMG resolved source geometry → eligible
- COOLR with interpretable location accuracy ≤ 5 km → eligible
- COOLR > 5 km → retained historically, but not used for fine-resolution point-feature training
- COOLR with unknown/uninterpretable location accuracy → retained historically, but excluded from point-feature training

This rule can be changed later, but every resulting count is recalculated here.


In [4]:
MAX_COOLR_ACCURACY_M = 5_000

def accuracy_to_m(value):
    if pd.isna(value):
        return np.nan

    text = str(value).strip().lower()

    if text in {"exact", "precise", "point"}:
        return 0.0

    match = re.search(r"(\d+(?:\.\d+)?)", text)
    if not match:
        return np.nan

    number = float(match.group(1))

    if "km" in text:
        return number * 1000.0
    if re.search(r"\bm\b", text):
        return number

    return np.nan


positive["location_accuracy_m"] = np.nan

coolr_mask = positive["source_dataset"].eq("NASA_COOLR")
jmg_mask = positive["source_dataset"].eq("JMG")

if "location_accuracy_source" not in positive.columns:
    positive["location_accuracy_source"] = pd.NA

positive.loc[coolr_mask, "location_accuracy_m"] = (
    positive.loc[coolr_mask, "location_accuracy_source"]
    .apply(accuracy_to_m)
)

# JMG coordinates came from resolved original source geometries.
positive.loc[jmg_mask, "location_accuracy_m"] = 0.0

positive["ml_eligible"] = False
positive["ml_exclusion_reason"] = pd.NA

positive.loc[jmg_mask, "ml_eligible"] = True

coolr_known = coolr_mask & positive["location_accuracy_m"].notna()

positive.loc[
    coolr_known & (positive["location_accuracy_m"] <= MAX_COOLR_ACCURACY_M),
    "ml_eligible"
] = True

positive.loc[
    coolr_known & (positive["location_accuracy_m"] > MAX_COOLR_ACCURACY_M),
    "ml_exclusion_reason"
] = "COOLR positional uncertainty > 5 km"

positive.loc[
    coolr_mask & positive["location_accuracy_m"].isna(),
    "ml_exclusion_reason"
] = "COOLR location accuracy unavailable/uninterpretable"

eligibility_summary = (
    positive.groupby(
        ["source_dataset", "ml_eligible"],
        dropna=False
    )
    .size()
    .reset_index(name="records")
)

display(eligibility_summary)

print("Full historical reviewed inventory:", len(positive))
print("Point-ML-eligible positives:", int(positive["ml_eligible"].sum()))
print("Not eligible for fine-resolution point ML:", int((~positive["ml_eligible"]).sum()))


,source_dataset,ml_eligible,records
0,JMG,True,52
1,NASA_COOLR,False,87
2,NASA_COOLR,True,148


Full historical reviewed inventory: 287
Point-ML-eligible positives: 200
Not eligible for fine-resolution point ML: 87


## 4. Date precision / temporal-feature readiness

This notebook does not fabricate missing dates.

The field below records what can safely be done later.

`event_date` from the cleaned inventory is parsed if available. If a source lacks a reliable date, it remains missing.


In [5]:
positive["event_date_parsed"] = pd.to_datetime(
    positive.get("event_date"),
    errors="coerce"
)

positive["event_year"] = positive["event_date_parsed"].dt.year
positive["event_month"] = positive["event_date_parsed"].dt.month
positive["event_day"] = positive["event_date_parsed"].dt.day

positive["date_precision_for_modelling"] = np.where(
    positive["event_date_parsed"].notna(),
    "day",
    "missing"
)

positive["annual_landcover_eligible"] = (
    positive["event_year"].notna()
)

positive["antecedent_rainfall_eligible"] = (
    positive["event_date_parsed"].notna()
)

date_summary = pd.DataFrame({
    "check": [
        "ML-eligible positives",
        "ML-eligible with usable event date",
        "ML-eligible without usable event date",
    ],
    "value": [
        int(positive["ml_eligible"].sum()),
        int((positive["ml_eligible"] & positive["event_date_parsed"].notna()).sum()),
        int((positive["ml_eligible"] & positive["event_date_parsed"].isna()).sum()),
    ]
})

display(date_summary)

year_distribution = (
    positive.loc[
        positive["ml_eligible"] & positive["event_year"].notna()
    ]
    .groupby("event_year")
    .size()
    .reset_index(name="records")
    .sort_values("event_year")
)

display(year_distribution)

year_distribution.to_csv(
    OUTPUT_DIR / "ml_eligible_year_distribution.csv",
    index=False
)


,check,value
0,ML-eligible positives,200
1,ML-eligible with usable event date,195
2,ML-eligible without usable event date,5


,event_year,records
0,2007.0,2
1,2008.0,4
2,2009.0,7
3,2010.0,5
4,2011.0,14
5,2012.0,9
6,2013.0,18
7,2014.0,24
8,2015.0,16
9,2016.0,18


## 5. Save historical vs ML-eligible inventories separately

Nothing is deleted from the historical inventory.


In [6]:
historical_audit_path = (
    OUTPUT_DIR / "positive_inventory_with_ml_eligibility.csv"
)

positive.drop(
    columns="geometry",
    errors="ignore"
).to_csv(
    historical_audit_path,
    index=False
)

eligible_positive = positive.loc[
    positive["ml_eligible"]
].copy()

eligible_geojson = (
    CLEAN_DIR / "malaysia_positive_inventory_ml_eligible.geojson"
)

eligible_csv = (
    CLEAN_DIR / "malaysia_positive_inventory_ml_eligible.csv"
)

eligible_export = eligible_positive.copy()
eligible_export["event_date_parsed"] = (
    eligible_export["event_date_parsed"]
    .astype("string")
    .replace("<NA>", None)
)

eligible_export.to_file(
    eligible_geojson,
    driver="GeoJSON"
)

eligible_positive.drop(
    columns="geometry",
    errors="ignore"
).to_csv(
    eligible_csv,
    index=False
)

print("Saved ML-eligible inventory:", eligible_geojson)


Saved ML-eligible inventory: C:\Landslide_FYP\Landslide_FYP_Development\02_cleaned_inventory\malaysia_positive_inventory_ml_eligible.geojson


## 6. Background candidate-generation rules

Generate a large candidate pool first.

**Do not label these as final pseudo-absences yet.**

Default:
- 3,000 background candidates
- 50 km equal-area sampling grid
- minimum 1 km candidate-to-candidate spacing
- positive exclusion buffer:
  - at least 2 km
  - COOLR uses its known positional uncertainty up to the accepted 5 km
  - JMG uses 2 km

Environmental predictors are deliberately not used to select only low-risk places.


In [7]:
SAMPLING_CRS = "EPSG:6933"

BACKGROUND_CANDIDATE_TARGET = 3000
GRID_SIZE_M = 50_000
MIN_BACKGROUND_SEPARATION_M = 1_000
MIN_POSITIVE_BUFFER_M = 2_000
RANDOM_SEED = 42

rng = np.random.default_rng(RANDOM_SEED)

eligible_m = eligible_positive.to_crs(SAMPLING_CRS)
malaysia_m = malaysia.to_crs(SAMPLING_CRS)

if len(malaysia_m) == 1:
    malaysia_geom_m = malaysia_m.geometry.iloc[0]
else:
    malaysia_geom_m = malaysia_m.dissolve().geometry.iloc[0]

eligible_m["sampling_exclusion_m"] = MIN_POSITIVE_BUFFER_M

eligible_coolr = eligible_m["source_dataset"].eq("NASA_COOLR")

eligible_m.loc[
    eligible_coolr,
    "sampling_exclusion_m"
] = np.maximum(
    MIN_POSITIVE_BUFFER_M,
    eligible_m.loc[eligible_coolr, "location_accuracy_m"]
)

buffers = [
    geom.buffer(float(distance))
    for geom, distance in zip(
        eligible_m.geometry,
        eligible_m["sampling_exclusion_m"]
    )
]

exclusion_union = unary_union(buffers)

sampling_area = malaysia_geom_m.difference(
    exclusion_union
)

malaysia_area_km2 = malaysia_geom_m.area / 1_000_000
sampling_area_km2 = sampling_area.area / 1_000_000

print(f"Malaysia area: {malaysia_area_km2:,.1f} km²")
print(f"Eligible background area: {sampling_area_km2:,.1f} km²")
print(f"Eligible share: {sampling_area_km2/malaysia_area_km2*100:.2f}%")


Malaysia area: 329,819.9 km²
Eligible background area: 326,104.5 km²
Eligible share: 98.87%


## 7. Build equal-area sampling grid

In [8]:
minx, miny, maxx, maxy = sampling_area.bounds

grid_geoms = []
grid_ids = []
counter = 1

x = math.floor(minx / GRID_SIZE_M) * GRID_SIZE_M

while x < maxx:
    y = math.floor(miny / GRID_SIZE_M) * GRID_SIZE_M

    while y < maxy:
        square = box(
            x,
            y,
            x + GRID_SIZE_M,
            y + GRID_SIZE_M
        )

        valid_part = square.intersection(sampling_area)

        if not valid_part.is_empty and valid_part.area > 0:
            grid_geoms.append(valid_part)
            grid_ids.append(f"GRID_{counter:04d}")
            counter += 1

        y += GRID_SIZE_M

    x += GRID_SIZE_M

grid = gpd.GeoDataFrame(
    {
        "grid_id": grid_ids,
        "geometry": grid_geoms,
    },
    geometry="geometry",
    crs=SAMPLING_CRS
)

grid["eligible_area_m2"] = grid.geometry.area

print("Eligible grid cells:", len(grid))


Eligible grid cells: 213


## 8. Allocate and generate 3,000 background candidates

In [9]:
def proportional_allocation(weights, total_n):
    weights = np.asarray(weights, dtype=float)

    raw = weights / weights.sum() * total_n
    base = np.floor(raw).astype(int)

    remaining = int(total_n - base.sum())

    if remaining > 0:
        frac = raw - base
        idx = np.argsort(-frac)[:remaining]
        base[idx] += 1

    return base


grid["target_n"] = proportional_allocation(
    grid["eligible_area_m2"],
    BACKGROUND_CANDIDATE_TARGET
)

print("Allocated target:", int(grid["target_n"].sum()))


Allocated target: 3000


In [10]:
def random_point(poly, rng):
    minx, miny, maxx, maxy = poly.bounds

    for _ in range(20_000):
        p = Point(
            rng.uniform(minx, maxx),
            rng.uniform(miny, maxy)
        )

        if poly.covers(p):
            return p

    return None


def far_enough(point, accepted, min_distance):
    for other in accepted:
        if point.distance(other) < min_distance:
            return False

    return True


accepted = []
rows = []

for _, row in grid.iterrows():

    required = int(row["target_n"])

    if required <= 0:
        continue

    geom = row[grid.geometry.name]
    made = 0
    attempts = 0

    while made < required and attempts < 100_000:
        attempts += 1

        candidate = random_point(geom, rng)

        if candidate is None:
            continue

        if not far_enough(
            candidate,
            accepted,
            MIN_BACKGROUND_SEPARATION_M
        ):
            continue

        accepted.append(candidate)

        rows.append({
            "background_id": f"BG_{len(accepted):05d}",
            "grid_id": row["grid_id"],
            "sampling_seed": RANDOM_SEED,
            "geometry": candidate,
        })

        made += 1

    if made < required:
        print(
            f"WARNING: {row['grid_id']} requested {required}, generated {made}"
        )


background_m = gpd.GeoDataFrame(
    rows,
    geometry="geometry",
    crs=SAMPLING_CRS
)

print("Generated background candidates:", len(background_m))

if len(background_m) != BACKGROUND_CANDIDATE_TARGET:
    raise RuntimeError(
        "Did not generate the complete background candidate pool."
    )


Generated background candidates: 3000


## 9. Candidate validity flags

At this stage, every candidate is:
- inside Malaysia,
- outside ML-positive exclusion buffers,
- spatially separated from other background candidates.

The following checks are intentionally deferred until the corresponding factor is extracted:

- valid DEM pixel
- open-water mask
- valid historical/current land-cover pixel
- soil raster coverage
- lithology coverage
- fault coverage

**Road distance, waterway distance, slope and elevation are not used as rejection criteria.**


In [11]:
background_m["inside_sampling_area"] = (
    background_m.geometry.apply(sampling_area.covers)
)

if not background_m["inside_sampling_area"].all():
    raise ValueError("Background point outside valid sampling area.")

background = background_m.to_crs("EPSG:4326")

background["longitude"] = background.geometry.x
background["latitude"] = background.geometry.y

background["sample_class"] = "background_candidate"
background["target"] = pd.NA

background["dem_valid"] = pd.NA
background["land_pixel_valid"] = pd.NA
background["factor_coverage_valid"] = pd.NA
background["final_pseudo_absence_selected"] = False

display(background.head())


,background_id,grid_id,sampling_seed,geometry,inside_sampling_area,longitude,latitude,sample_class,target,dem_valid,land_pixel_valid,factor_coverage_valid,final_pseudo_absence_selected
0,BG_00001,GRID_0001,42,POINT (99.85213 6.25585),True,99.852134,6.255855,background_candidate,<NA>,<NA>,<NA>,<NA>,False
1,BG_00002,GRID_0002,42,POINT (99.67979 6.37066),True,99.679791,6.370659,background_candidate,<NA>,<NA>,<NA>,<NA>,False
2,BG_00003,GRID_0002,42,POINT (99.8371 6.44302),True,99.837096,6.443016,background_candidate,<NA>,<NA>,<NA>,<NA>,False
3,BG_00004,GRID_0002,42,POINT (99.77596 6.3273),True,99.775958,6.327303,background_candidate,<NA>,<NA>,<NA>,<NA>,False
4,BG_00005,GRID_0002,42,POINT (99.89314 6.40588),True,99.893144,6.405882,background_candidate,<NA>,<NA>,<NA>,<NA>,False


## 10. Save candidate pool

This output replaces the old final pseudo-absence files for the revised workflow.


In [12]:
background_geojson = (
    CLEAN_DIR / "background_candidates_3000.geojson"
)

background_csv = (
    CLEAN_DIR / "background_candidates_3000.csv"
)

background.to_file(
    background_geojson,
    driver="GeoJSON"
)

background.drop(
    columns="geometry",
    errors="ignore"
).to_csv(
    background_csv,
    index=False
)

print("Saved:", background_geojson)
print("Saved:", background_csv)


Saved: C:\Landslide_FYP\Landslide_FYP_Development\02_cleaned_inventory\background_candidates_3000.geojson
Saved: C:\Landslide_FYP\Landslide_FYP_Development\02_cleaned_inventory\background_candidates_3000.csv


## 11. Map revised ML-eligible positives and background candidates


In [13]:
fig, ax = plt.subplots(figsize=(12, 9))

malaysia.boundary.plot(
    ax=ax,
    linewidth=0.8
)

background.plot(
    ax=ax,
    markersize=5,
    alpha=0.35,
    label="Background candidates"
)

eligible_positive.plot(
    ax=ax,
    marker="x",
    markersize=32,
    alpha=0.8,
    label="Point-ML-eligible landslides"
)

ax.set_title(
    "Revised Point-ML Positives and Background Candidate Pool"
)
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.legend()
ax.grid(True, alpha=0.2)

plt.tight_layout()

map_path = (
    OUTPUT_DIR / "revised_background_candidate_map.png"
)

fig.savefig(
    map_path,
    dpi=180,
    bbox_inches="tight"
)

plt.close(fig)

print("Saved:", map_path)


Saved: C:\Landslide_FYP\Landslide_FYP_Development\01_preprocessing\stage04_revised_outputs\revised_background_candidate_map.png


## 12. Feature-use register for the next stages

This table prevents temporal misuse of predictors.


In [14]:
feature_register = pd.DataFrame([
    {
        "feature": "elevation",
        "temporal_type": "quasi-static",
        "candidate_rejection_rule": "reject only NoData/invalid pixel",
        "model_use": "baseline"
    },
    {
        "feature": "slope",
        "temporal_type": "quasi-static",
        "candidate_rejection_rule": "reject only NoData/invalid pixel",
        "model_use": "baseline"
    },
    {
        "feature": "aspect",
        "temporal_type": "quasi-static",
        "candidate_rejection_rule": "reject only NoData/invalid pixel",
        "model_use": "baseline"
    },
    {
        "feature": "soil",
        "temporal_type": "quasi-static",
        "candidate_rejection_rule": "reject only missing required coverage",
        "model_use": "baseline if coverage valid"
    },
    {
        "feature": "lithology",
        "temporal_type": "relatively static",
        "candidate_rejection_rule": "do not select low-risk classes; require valid coverage",
        "model_use": "baseline if national coverage valid"
    },
    {
        "feature": "distance_to_fault",
        "temporal_type": "relatively static",
        "candidate_rejection_rule": "not a rejection criterion",
        "model_use": "baseline if coverage valid"
    },
    {
        "feature": "distance_to_waterway",
        "temporal_type": "time-sensitive / imperfect history",
        "candidate_rejection_rule": "not a rejection criterion",
        "model_use": "baseline-with-limitation or enhanced model"
    },
    {
        "feature": "road_distance",
        "temporal_type": "time-sensitive",
        "candidate_rejection_rule": "not a rejection criterion",
        "model_use": "historical road snapshot where defensible"
    },
    {
        "feature": "land_cover",
        "temporal_type": "time-sensitive",
        "candidate_rejection_rule": "water/invalid pixel may be rejected; do not reject steep/forest/crop etc.",
        "model_use": "match source year to event/reference year"
    },
    {
        "feature": "rainfall",
        "temporal_type": "event-time sensitive",
        "candidate_rejection_rule": "not used to choose easy negatives",
        "model_use": "date-matched enhanced model"
    },
])

display(feature_register)

feature_register.to_csv(
    OUTPUT_DIR / "feature_use_register.csv",
    index=False
)


,feature,temporal_type,candidate_rejection_rule,model_use
0,elevation,quasi-static,reject only NoData/invalid pixel,baseline
1,slope,quasi-static,reject only NoData/invalid pixel,baseline
2,aspect,quasi-static,reject only NoData/invalid pixel,baseline
3,soil,quasi-static,reject only missing required coverage,baseline if coverage valid
4,lithology,relatively static,do not select low-risk classes; require valid ...,baseline if national coverage valid
5,distance_to_fault,relatively static,not a rejection criterion,baseline if coverage valid
6,distance_to_waterway,time-sensitive / imperfect history,not a rejection criterion,baseline-with-limitation or enhanced model
7,road_distance,time-sensitive,not a rejection criterion,historical road snapshot where defensible
8,land_cover,time-sensitive,water/invalid pixel may be rejected; do not re...,match source year to event/reference year
9,rainfall,event-time sensitive,not used to choose easy negatives,date-matched enhanced model


## 13. Final Stage-4 revised summary


In [15]:
stage04_summary = pd.DataFrame({
    "check": [
        "Reviewed historical positives",
        "Point-ML-eligible positives",
        "Excluded from fine-resolution point ML",
        "COOLR maximum accepted accuracy m",
        "Background candidate target",
        "Background candidates generated",
        "Eligible sampling area percent",
        "Final pseudo-absences selected at this stage",
    ],
    "value": [
        len(positive),
        int(positive["ml_eligible"].sum()),
        int((~positive["ml_eligible"]).sum()),
        MAX_COOLR_ACCURACY_M,
        BACKGROUND_CANDIDATE_TARGET,
        len(background),
        round(sampling_area_km2 / malaysia_area_km2 * 100, 2),
        0,
    ]
})

display(stage04_summary)

stage04_summary.to_csv(
    OUTPUT_DIR / "stage04_revised_summary.csv",
    index=False
)

print()
print("Stage 4 revised workflow complete.")
print("The 3,000 points are BACKGROUND CANDIDATES, not final pseudo-absences.")
print()
print("NEXT:")
print("05_feature_extraction_and_candidate_validity.ipynb")


,check,value
0,Reviewed historical positives,287.00
1,Point-ML-eligible positives,200.00
2,Excluded from fine-resolution point ML,87.00
3,COOLR maximum accepted accuracy m,5000.00
4,Background candidate target,3000.00
5,Background candidates generated,3000.00
6,Eligible sampling area percent,98.87
7,Final pseudo-absences selected at this stage,0.00



Stage 4 revised workflow complete.
The 3,000 points are BACKGROUND CANDIDATES, not final pseudo-absences.

NEXT:
05_feature_extraction_and_candidate_validity.ipynb


# Outputs

```text
Landslide_FYP_Development/
├── 01_preprocessing/
│   └── stage04_revised_outputs/
│       ├── positive_inventory_with_ml_eligibility.csv
│       ├── ml_eligible_year_distribution.csv
│       ├── revised_background_candidate_map.png
│       ├── feature_use_register.csv
│       └── stage04_revised_summary.csv
│
└── 02_cleaned_inventory/
    ├── malaysia_positive_inventory_ml_eligible.geojson
    ├── malaysia_positive_inventory_ml_eligible.csv
    ├── background_candidates_3000.geojson
    └── background_candidates_3000.csv
```

The old `pseudo_absence_master_*` and `modelling_samples_1to*` outputs should not be used in the revised pipeline.
